# ECG Signal Analysis

This notebook analyzes the ECG signal from the MIT-BIH database (files `b02.dat` and `b02.qrs`).
Analysis includes:
- Signal visualization with R-peak annotations.
- ECG and RR interval histograms.
- RR Tachogram.
- Poincare Plot.
- Morlet Wavelet Transform (Continuous Wavelet Transform).

In [ ]:
import wfdb
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import os

# File path settings
record_name = 'b02'
lab_folder = '/Users/divyanshukaushik/project(0001)/lab'
file_path = os.path.join(lab_folder, record_name)

# Initialize variables
try:
    # Load signal
    record = wfdb.rdrecord(file_path)
    sig = record.p_signal[:, 0]
    fs = record.fs
    # Load annotations
    annotation = wfdb.rdann(file_path, 'qrs')
    r_peaks = annotation.sample
    
    rr_intervals = np.diff(r_peaks) / fs * 1000 # in ms
    time = np.arange(len(sig)) / fs
    
    print(f"Loaded {record_name} successfully.")
    print(f"Signal Length: {len(sig)} samples")
    print(f"R-peaks: {len(r_peaks)} detected")
except Exception as e:
    print(f"Error loading data: {e}")
    # Stop if data cannot be loaded
    raise e

## 1. ECG Signal with R-peak Annotations

In [ ]:
start_idx, end_idx = 0, min(3000, len(sig))
plt.figure(figsize=(15, 5))
plt.plot(time[start_idx:end_idx], sig[start_idx:end_idx], label='ECG Signal', color='#2E86C1')
peaks_in_segment = r_peaks[(r_peaks >= start_idx) & (r_peaks < end_idx)]
plt.scatter(time[peaks_in_segment], sig[peaks_in_segment], color='red', marker='x', label='R-peaks')
plt.title('ECG Signal with R-peak Annotations')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (mV)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Histograms

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].hist(sig, bins=100, color='#85C1E9', edgecolor='black', alpha=0.7)
ax[0].set_title('Histogram of ECG Amplitude')
ax[0].set_xlabel('Amplitude (mV)')
ax[0].set_ylabel('Frequency')
ax[1].hist(rr_intervals, bins=50, color='#F7DC6F', edgecolor='black', alpha=0.7)
ax[1].set_title('Histogram of RR Intervals')
ax[1].set_xlabel('RR Interval (ms)')
ax[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## 3. Tachogram

In [ ]:
plt.figure(figsize=(15, 5))
rr_times = r_peaks[1:] / fs
plt.step(rr_times, rr_intervals, where='post', color='#239B56')
plt.title('RR Tachogram')
plt.xlabel('Time (s)')
plt.ylabel('RR Interval (ms)')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Poincare Plot

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(rr_intervals[:-1], rr_intervals[1:], s=5, alpha=0.5, color='#E74C3C')
plt.title('Poincare Plot')
plt.xlabel('RR(n) (ms)')
plt.ylabel('RR(n+1) (ms)')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

## 5. Morlet Wavelet Transform

In [ ]:
seg_start, seg_end = 0, min(1000, len(sig))
sig_segment = sig[seg_start:seg_end]
widths = np.arange(1, 31)
cwtmatr = signal.cwt(sig_segment, signal.morlet2, widths)
plt.figure(figsize=(15, 6))
plt.imshow(np.abs(cwtmatr), extent=[0, len(sig_segment)/fs, 1, 31], cmap='jet', aspect='auto', vmax=abs(cwtmatr).max(), vmin=0)
plt.title('Morlet Wavelet Scaleogram')
plt.ylabel('Scale')
plt.xlabel('Time (s)')
plt.colorbar(label='Magnitude')
plt.show()